In [ ]:
import pickle
from pathlib import Path

def load_inference_cache(load_path='inference_cache.pkl'):
    if Path(load_path).exists():
        with open(load_path, 'rb') as f:
            cache = pickle.load(f)
        print(f"Cache loaded from {load_path}")
        return cache
    else:
        print(f"Cache file not found: {load_path}")
        return None


In [ ]:
inference_cache = load_inference_cache('inference_cache.pkl')

In [ ]:
import torch
import torchvision.models
from torchvision.datasets import ImageNet
import torch.nn as nn
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision.transforms.functional import to_pil_image
from torchvision import transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw
import os
from torch.utils.data import Dataset, DataLoader, Subset


device = torch.device("cuda")

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors_darker = sns.color_palette("dark", 5 * 2)
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
class LinearClassifier(nn.Module):
    """Linear layer to train on top of frozen features"""

    def __init__(self, dim, num_labels=1000):
        super(LinearClassifier, self).__init__()
        self.num_labels = num_labels
        self.linear = nn.Linear(dim, num_labels)
        self.linear.weight.data.normal_(mean=0.0, std=0.01)
        self.linear.bias.data.zero_()

    def forward(self, x):
        # flatten
        x = x.view(x.size(0), -1)

        # linear layer
        return self.linear(x)

In [ ]:
def get_bbox_area_ratio_after_crop(xml_path, original_width, original_height, crop_size=224):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Resize
    if original_width < original_height:
        resize_width = 256
        resize_height = int(original_height * 256 / original_width)
    else:
        resize_height = 256
        resize_width = int(original_width * 256 / original_height)

    crop_left = (resize_width - crop_size) // 2
    crop_top = (resize_height - crop_size) // 2
    crop_right = crop_left + crop_size
    crop_bottom = crop_top + crop_size

    mask = np.zeros((crop_size, crop_size), dtype=bool)

    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        scale_x = resize_width / original_width
        scale_y = resize_height / original_height

        xmin_resized = xmin * scale_x
        ymin_resized = ymin * scale_y
        xmax_resized = xmax * scale_x
        ymax_resized = ymax * scale_y

        intersect_xmin = max(xmin_resized, crop_left)
        intersect_ymin = max(ymin_resized, crop_top)
        intersect_xmax = min(xmax_resized, crop_right)
        intersect_ymax = min(ymax_resized, crop_bottom)

        if intersect_xmin < intersect_xmax and intersect_ymin < intersect_ymax:
            x1 = int(intersect_xmin - crop_left)
            y1 = int(intersect_ymin - crop_top)
            x2 = int(intersect_xmax - crop_left)
            y2 = int(intersect_ymax - crop_top)

            mask[y1:y2, x1:x2] = True

    object_pixels = np.sum(mask)
    crop_area = crop_size * crop_size

    return object_pixels / crop_area

In [ ]:
def load_ddp_checkpoint(model, checkpoint_path):
    state_dict = torch.load(checkpoint_path)
    new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    return model

def parse_bbox_annotation(xml_path):
    if not os.path.exists(xml_path):
        return []

    tree = ET.parse(xml_path)
    root = tree.getroot()

    bboxes = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        bboxes.append([xmin, ymin, xmax, ymax])

    return bboxes

def mask_object_regions(image_path, xml_path):
    img = Image.open(image_path).convert('RGB')

    bboxes = parse_bbox_annotation(xml_path)

    if bboxes:
        draw = ImageDraw.Draw(img)
        for bbox in bboxes:
            xmin, ymin, xmax, ymax = bbox
            draw.rectangle([xmin, ymin, xmax, ymax], fill='black')

    return img

val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=3),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

def denormalize(tensor):
    mean = torch.tensor((0.485, 0.456, 0.406)).view(3, 1, 1).to(tensor.device)
    std = torch.tensor((0.229, 0.224, 0.225)).view(3, 1, 1).to(tensor.device)
    denorm = tensor * std + mean
    return torch.clamp(denorm, 0, 1)

dataset_test = ImageNet("/home/dataset/imagenet", split="val", transform=val_transform)
bbox_dir = "/home/dataset/imagenet/ILSVRC/Annotations/CLS-LOC/val"

In [ ]:
steps = ["0_200", "2_1400", "5_3400", "10_6800", "20_13400", "30_20200", "40_26800", "50_33400", "60_40200", "70_46800", "80_53600", "99_66799"]
steps_int = [int(s.split('_')[1]) for s in steps]
models_steps = []
linear_steps = []
for step in steps:
    resnet = torchvision.models.resnet50(weights=None)
    embed_dim = resnet.fc.weight.shape[1]
    resnet.fc = nn.Identity()
    resnet.load_state_dict(torch.load(f"/models/jhkim/bt-imagenet/{step}.pth"))
    models_steps.append(resnet)
    linear = LinearClassifier(embed_dim)
    load_ddp_checkpoint(linear, f"/models/jhkim/bt-imagenet/{step}/final_checkpoint.pth")
    linear_steps.append(linear)

In [ ]:
class_data = {}
all_data_list = []

print("Collecting images with bounding boxes for all classes...")
for idx in tqdm(range(len(dataset_test))):
    image_path, label = dataset_test.samples[idx]

    if label not in class_data:
        class_data[label] = []

    img_name = os.path.basename(image_path)
    xml_name = os.path.splitext(img_name)[0] + ".xml"
    xml_path = os.path.join(bbox_dir, xml_name)

    if os.path.exists(xml_path):
        img = Image.open(image_path)
        img_width, img_height = img.size

        bbox_ratio = get_bbox_area_ratio_after_crop(xml_path, img_width, img_height, crop_size=224)

        class_data[label].append({
            'idx': idx,
            'image_path': image_path,
            'xml_path': xml_path,
            'bbox_ratio': bbox_ratio
        })
        all_data_list.append({
            'image_path': image_path,
            'xml_path': xml_path,
            'class': label,
        })

valid_classes = {cls: data for cls, data in class_data.items() if len(data) >= 10}
print(f"Found {len(valid_classes)} classes with at least 10 images with bboxes")

class_bbox_stats = {}
for cls, data in valid_classes.items():
    bbox_ratios = [d['bbox_ratio'] for d in data]
    class_bbox_stats[cls] = {
        'mean_bbox_ratio': np.mean(bbox_ratios),
        'n_samples': len(data)
    }

bbox_sizes = [class_bbox_stats[cls]['mean_bbox_ratio'] for cls in valid_classes.keys()]
bbox_percentiles = np.percentile(bbox_sizes, [5, 10, 15, 20])

class_bbox_group = {}
for cls in valid_classes.keys():
    bbox_ratio = class_bbox_stats[cls]['mean_bbox_ratio']
    if bbox_percentiles[0] <= bbox_ratio < bbox_percentiles[1]:
        class_bbox_group[cls] = 'Small'
    elif bbox_percentiles[1] <= bbox_ratio < bbox_percentiles[2]:
        class_bbox_group[cls] = 'Medium'
    elif bbox_percentiles[2] <= bbox_ratio < bbox_percentiles[3]:
        class_bbox_group[cls] = 'Large'
    else:
        class_bbox_group[cls] = 'Other'

all_images = []
for cls, data_list in valid_classes.items():
    for d in data_list:
        all_images.append({
            'idx': d['idx'],
            'class': cls,
            'bbox_ratio': d['bbox_ratio']
        })

In [ ]:
class MaskedImageDataset(Dataset):
    def __init__(self, base_dataset, data_list, transform):
        self.base_dataset = base_dataset
        self.data_list = data_list
        self.transform = transform
        self.preprocess()

    def preprocess(self):
        print("Preprocessing bboxes...")
        for data in tqdm(self.data_list):
            img = Image.open(data['image_path'])
            orig_w, orig_h = img.size

            bboxes = parse_bbox_annotation(data['xml_path'])

            data['orig_size'] = (orig_w, orig_h)
            data['bboxes'] = bboxes

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        data = self.data_list[idx]

        img = Image.open(data['image_path']).convert('RGB')
        orig_w, orig_h = data['orig_size']
        bboxes = data['bboxes']

        # Apply transform to image
        img_tensor = self.transform(img)
        H, W = img_tensor.shape[1], img_tensor.shape[2]  # 224, 224

        # Create bbox mask on PIL Image, then transform it
        bbox_mask_pil = Image.new('L', (orig_w, orig_h), 0)
        draw = ImageDraw.Draw(bbox_mask_pil)

        for bbox in bboxes:
            xmin, ymin, xmax, ymax = bbox
            draw.rectangle([xmin, ymin, xmax, ymax], fill=255)

        # Apply same transform to mask (without normalization)
        mask_transform = transforms.Compose([
            transforms.Resize(256, interpolation=3),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
        ])

        bbox_mask_tensor = mask_transform(bbox_mask_pil)  # (1, 224, 224)
        bbox_mask_tensor = (bbox_mask_tensor > 0.5).float()  # Binarize

        # Create 3-channel mask
        mask = 1 - bbox_mask_tensor.repeat(3, 1, 1)  # 0 where bbox, 1 elsewhere

        object_tensor = img_tensor * (1 - mask)
        background_tensor = img_tensor * mask

        return img_tensor, object_tensor, background_tensor, data['class'], mask, idx

In [ ]:
masking_dataset = MaskedImageDataset(dataset_test, all_data_list, val_transform)
masking_loader = DataLoader(masking_dataset, batch_size=512, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)
N = len(masking_dataset)

Stop for inference


In [ ]:
results = {step: {} for step in steps}

print("Calculating global class statistics...")
for step in steps:
    class_stats = {cls: {'orig_top1': 0, 'orig_top5': 0,
                         'object_top1': 0, 'object_top5': 0,
                         'background_top1': 0, 'background_top5': 0,
                         'total': 0}
                   for cls in valid_classes.keys()}

    step_data = inference_cache[step]

    for idx, data in step_data.items():
        cls = data['class']
        if cls not in class_stats: continue

        class_stats[cls]['orig_top1'] += int(data['orig_top1'])
        class_stats[cls]['orig_top5'] += int(data['orig_top5'])
        class_stats[cls]['object_top1'] += int(data['obj_top1'])
        class_stats[cls]['object_top5'] += int(data['obj_top5'])
        class_stats[cls]['background_top1'] += int(data['bg_top1'])
        class_stats[cls]['background_top5'] += int(data['bg_top5'])
        class_stats[cls]['total'] += 1

    for cls, stats in class_stats.items():
        if stats['total'] > 0:
            results[step][cls] = {
                k: v / stats['total'] for k, v in stats.items() if k != 'total'
            }
            results[step][cls]['mean_bbox_ratio'] = class_bbox_stats[cls]['mean_bbox_ratio']

In [ ]:
import pandas as pd



last_step = steps[-1]
step_data = inference_cache[last_step]

data_for_plot = []

class_stats = {}
for idx, data in step_data.items():
    cls = data['class']
    if cls not in class_stats:
        class_stats[cls] = {'bg_corr': 0, 'obj_corr': 0, 'total': 0}

    class_stats[cls]['bg_corr'] += int(data['bg_top1'])
    class_stats[cls]['obj_corr'] += int(data['obj_top1'])
    class_stats[cls]['total'] += 1

for cls, stats in class_stats.items():
    if stats['total'] < 10: continue

    bg_acc = stats['bg_corr'] / stats['total'] * 100
    obj_acc = stats['obj_corr'] / stats['total'] * 100
    diff = bg_acc - obj_acc

    bbox_ratio = class_bbox_stats[cls]['mean_bbox_ratio']

    data_for_plot.append({
        'Class': cls,
        'Diff': diff,
        'BBox Ratio': bbox_ratio
    })

df = pd.DataFrame(data_for_plot)
df = df.sort_values('Diff', ascending=False).reset_index(drop=True)

plt.figure(figsize=(15, 6))
points = plt.scatter(df.index, df['Diff'], c=df['BBox Ratio'], cmap='coolwarm', alpha=0.5, s=30)
plt.colorbar(points, label='Mean BBox Ratio (Red=Large BG)')

plt.axhline(0, color='gray', linestyle='--')
plt.title(f'Difference between Background and Object Accuracy (Top-1) per Class (Step {last_step})', fontsize=15)
plt.xlabel('Class Rank (Sorted by Background Bias)', fontsize=12)
plt.ylabel('Background Acc - Object Acc (%)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

last_step = steps[-1]
step_data = inference_cache[last_step]

data_for_plot = []

class_stats = {}
for idx, data in step_data.items():
    cls = data['class']
    if cls not in class_stats:
        class_stats[cls] = {'bg_corr': 0, 'obj_corr': 0, 'total': 0}

    class_stats[cls]['bg_corr'] += int(data['bg_top5'])
    class_stats[cls]['obj_corr'] += int(data['obj_top5'])
    class_stats[cls]['total'] += 1

for cls, stats in class_stats.items():
    if stats['total'] < 10: continue

    bg_acc = stats['bg_corr'] / stats['total'] * 100
    obj_acc = stats['obj_corr'] / stats['total'] * 100
    diff = bg_acc - obj_acc

    bbox_ratio = class_bbox_stats[cls]['mean_bbox_ratio']

    data_for_plot.append({
        'Class': cls,
        'Diff': diff,
        'BBox Ratio': bbox_ratio
    })

df = pd.DataFrame(data_for_plot)
df = df.sort_values('Diff', ascending=False).reset_index(drop=True)

plt.figure(figsize=(15, 6))
points = plt.scatter(df.index, df['Diff'], c=df['BBox Ratio'], cmap='coolwarm', alpha=0.5, s=30)
plt.colorbar(points, label='Mean BBox Ratio (Red=Large BG)')

plt.axhline(0, color='gray', linestyle='--')
plt.title(f'Difference between Background and Object Accuracy (Top-5) per Class (Step {last_step})', fontsize=15)
plt.xlabel('Class Rank (Sorted by Background Bias)', fontsize=12)
plt.ylabel('Background Acc - Object Acc (%)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

target_epochs = range(len(steps))
epoch_names = {k: steps_int[k] for k in target_epochs}

all_plot_data = []

for epoch_idx in target_epochs:
    try:
        step_key = steps[epoch_idx]
    except (IndexError, KeyError):
        continue

    step_data = inference_cache[step_key]
    step_value = epoch_names.get(epoch_idx, epoch_idx)

    for idx, data in step_data.items():
        cls = data['class']

        if cls not in class_data:
            continue

        sample_info = None
        for sample in class_data[cls]:
            if sample['idx'] == idx:
                sample_info = sample
                break
        bbox_ratio = sample_info['bbox_ratio']

        if bbox_ratio == 0.0:
            continue

        bg_acc = int(data['bg_top5'])
        obj_acc = int(data['obj_top5'])
        diff_5 = (bg_acc - obj_acc) * 100  # 0 or 100

        all_plot_data.append({
            'Step': step_value,
            'Class': cls,
            'Sample_Idx': idx,
            'Diff_Top5': diff_5,
            'BBox Ratio': bbox_ratio
        })

df = pd.DataFrame(all_plot_data)

n_bins = 25
df['BBox Bin'] = pd.cut(df['BBox Ratio'], bins=n_bins, labels=False)
bin_edges = pd.cut(df['BBox Ratio'], bins=n_bins, retbins=True)[1]

pivot_top5 = df.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='mean')

step_values = sorted(pivot_top5.columns.values)
pivot_top5 = pivot_top5[step_values]

x_edges = np.concatenate([[0], step_values])
y_edges = bin_edges

fig, ax = plt.subplots(figsize=(14, 10))

im = ax.pcolormesh(x_edges, y_edges, pivot_top5.values, cmap='RdBu_r', vmin=-60, vmax=60)
ax.set_xlabel('Step')
ax.set_ylabel('Object Ratio')
plt.colorbar(im, ax=ax, label='Top-5 Spuriosity')
cbar.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('figure/spuriosity_orig.pdf')
plt.show()

print(f"Total data points: {len(df)} (= ~50k samples × {len(step_values)} steps)")
print(f"Samples per step (avg): {len(df) / len(step_values):.1f}")
print(f"Unique samples: {df['Sample_Idx'].nunique()}")
print(f"Unique classes: {df['Class'].nunique()}")
print(f"BBox Ratio range: {df['BBox Ratio'].min():.4f} ~ {df['BBox Ratio'].max():.4f}")

In [ ]:
n_bins = 25
df['BBox Bin'] = pd.cut(df['BBox Ratio'], bins=n_bins, labels=False)
bin_edges = pd.cut(df['BBox Ratio'], bins=n_bins, retbins=True)[1]
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# ----------------------------------------------------------
# 1) Per-bin sample count
# ----------------------------------------------------------
one_step = df[df['Step'] == df['Step'].unique()[0]]
bin_counts = one_step.groupby('BBox Bin').size()

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(bin_centers[bin_counts.index], bin_counts.values,
       width=(bin_edges[1] - bin_edges[0]) * 0.85,
       color='#5B9BD5', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Object Ratio')
ax.set_ylabel('Samples')
ax.set_title('Per-Bin Sample Count')
plt.tight_layout()
plt.savefig('figure/perbin_sample_count.pdf')
plt.show()

print("Per-bin sample counts:")
for b in sorted(bin_counts.index):
    lo, hi = bin_edges[b], bin_edges[b + 1]
    print(f"  Bin {b:2d}  [{lo:.3f}, {hi:.3f})  n = {bin_counts[b]}")
print(f"  Total: {bin_counts.sum()}")

# ----------------------------------------------------------
# 2) Pivot tables: mean, count, sem
# ----------------------------------------------------------
pivot_mean  = df.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='mean')
pivot_count = df.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='count')
pivot_sem   = df.pivot_table(values='Diff_Top5', index='BBox Bin', columns='Step', aggfunc='sem')

step_values = sorted(pivot_mean.columns.values)
pivot_mean  = pivot_mean[step_values]
pivot_count = pivot_count[step_values]
pivot_sem   = pivot_sem[step_values]

x_edges = np.concatenate([[0], step_values])
y_edges = bin_edges

# ----------------------------------------------------------
# 3) 3-panel figure: mean heatmap | SEM heatmap | sample count
# ----------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 8),
                          gridspec_kw={'width_ratios': [4, 4, 1.2]})

# (a) Spuriosity heatmap (same as Fig. 7)
im0 = axes[0].pcolormesh(x_edges, y_edges, pivot_mean.values,
                          cmap='RdBu_r', vmin=-60, vmax=60)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Object Ratio')
axes[0].set_title('Top-5 Spuriosity (Mean)')
plt.colorbar(im0, ax=axes[0], label='Spuriosity')

# (b) SEM heatmap
im1 = axes[1].pcolormesh(x_edges, y_edges, pivot_sem.values,
                          cmap='Oranges', vmin=0)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Object Ratio')
axes[1].set_title('Standard Error of Mean')
plt.colorbar(im1, ax=axes[1], label='SEM')

# (c) Per-bin sample count (horizontal bar)
axes[2].barh(bin_centers[bin_counts.index], bin_counts.values,
             height=(bin_edges[1] - bin_edges[0]) * 0.85,
             color='#5B9BD5', edgecolor='white', linewidth=0.5)
axes[2].set_xlabel('Samples')
axes[2].set_ylabel('Object Ratio')
axes[2].set_title('n per bin')
axes[2].set_ylim(y_edges[0], y_edges[-1])

plt.tight_layout()
plt.savefig('figure/spuriosity_with_uncertainty.png')
plt.show()

In [ ]:
sorted_classes = sorted(class_bbox_stats.keys(), key=lambda c: class_bbox_stats[c]['mean_bbox_ratio'])

for i, cls in enumerate(sorted_classes[:10]):
    print(f"{i+1}. Class {cls}: {class_bbox_stats[cls]['mean_bbox_ratio']:.3f}")

for i, cls in enumerate(sorted_classes[:10]):
    print(f"{i+1}. {dataset_test.classes[cls]}: {class_bbox_stats[cls]['mean_bbox_ratio']:.3f}")

In [ ]:
masking_dataset = MaskedImageDataset(dataset_test, all_data_list, val_transform)
masking_loader = DataLoader(masking_dataset, batch_size=128, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)
N = len(masking_dataset)

In [ ]:
transition_success = []

vis_steps = [steps[1], steps[3], steps[-1]]

# load model
step_idx = steps.index(vis_steps[0])
model = models_steps[step_idx].to(device).eval()
linear = linear_steps[step_idx].to(device).eval()
target_layers = [model.layer4[-1]]
model_grad = nn.Sequential(model, linear)

for orig_batch, object_batch, background_batch, classes_label, bbox_mask, idx_batch in tqdm(masking_loader):
    batch_size = orig_batch.size(0)

    orig_batch = orig_batch.to(device)

    for batch_idx in range(batch_size):
        i = idx_batch[batch_idx].item()
        first_cache = inference_cache[vis_steps[0]][i]

        cond1 = first_cache['bg_top5'] and (not first_cache['obj_top5'])

        mid_cache = inference_cache[vis_steps[1]][i]
        cond2 = mid_cache['bg_top5']

        last_cache = inference_cache[vis_steps[-1]][i]
        cond3 = (not last_cache['bg_top5']) and last_cache['obj_top5']

        if cond1 and cond2 and cond3:
            transition_success.append({
                'idx': i,
                'class': last_cache['class'],
                'bbox_ratio': all_images[i]['bbox_ratio'],
                'conf_orig_last': last_cache['conf_orig'],
                'conf_obj_last': last_cache['conf_obj'],
                'conf_bg_last': last_cache['conf_bg'],
            })


print(f"Total transition_success samples: {len(transition_success)}")

BBOX_THRESHOLD = 0.3
BBOX_THRESHOLD_min = 0.05

filtered_success = [s for s in transition_success if (s['bbox_ratio'] <= BBOX_THRESHOLD and s['bbox_ratio'] > BBOX_THRESHOLD_min)]

print(f"{BBOX_THRESHOLD_min} < BBox <= {BBOX_THRESHOLD}")
print(f"  Transition Success: {len(filtered_success)} samples")

In [ ]:
class_counts = {}
selected_samples = []
max_per_class = 2
max_total = 8

ignore_class = [85, 319, 320, 654, 673, 780, 964, 968, 538, 680, 49, 405, 736]

for s in filtered_success:
    cls = s['class']
    if cls in ignore_class:
        continue
    if cls not in class_counts:
        class_counts[cls] = 0

    if class_counts[cls] < max_per_class and len(selected_samples) < max_total:
        s['transition'] = 'success'
        selected_samples.append(s)
        class_counts[cls] += 1

    if len(selected_samples) >= max_total:
        break

print(f"\nSelected {len(selected_samples)} samples:")
for s in selected_samples:
    print(f"  idx={s['idx']}, class={s['class']}, bbox_ratio={s['bbox_ratio']:.3f}")

print(f"\nClass distribution:")
for cls, count in class_counts.items():
    print(f"  Class {cls}: {count} samples")

from matplotlib.patches import Rectangle

n_samples = len(selected_samples)
n_steps = len(vis_steps)

fig, axes = plt.subplots(n_steps + 1, n_samples, figsize=(n_samples * 3, (n_steps + 1) * 3.5))

if n_samples == 1:
    axes = axes.reshape(-1, 1)

for col, sample in enumerate(selected_samples):
    idx = sample['idx']
    cls = sample['class']
    bbox_ratio = sample['bbox_ratio']

    orig_batch, object_batch, background_batch, classes_label, _, _ = masking_dataset[idx]

    img_display = denormalize(orig_batch)
    img_np = img_display.permute(1, 2, 0).cpu().numpy()

    print(img_np.shape)

    axes[0, col].imshow(img_np)
    axes[0, col].set_title(f"Class: {cls}\nBBox Ratio: {bbox_ratio:.5f}",
                           fontsize=11, fontweight='bold')
    axes[0, col].axis('off')

    image_path, _ = dataset_test.samples[idx]
    img_name = os.path.basename(image_path)
    xml_name = os.path.splitext(img_name)[0] + ".xml"
    xml_path = os.path.join(bbox_dir, xml_name)

    if os.path.exists(xml_path):
        tree = ET.parse(xml_path)
        root = tree.getroot()

        size = root.find('size')
        orig_w = int(size.find('width').text)
        orig_h = int(size.find('height').text)

        # CenterCrop 224
        crop_size = 224

        # after Resize
        if orig_w < orig_h:
            resize_w = 256
            resize_h = int(orig_h * 256 / orig_w)
        else:
            resize_h = 256
            resize_w = int(orig_w * 256 / orig_h)

        # CenterCrop offset
        crop_top = (resize_h - crop_size) // 2
        crop_left = (resize_w - crop_size) // 2

        for obj in root.findall('object'):
            bbox = obj.find('bndbox')
            xmin = int(bbox.find('xmin').text)
            ymin = int(bbox.find('ymin').text)
            xmax = int(bbox.find('xmax').text)
            ymax = int(bbox.find('ymax').text)

            # 1. original → Resize
            xmin_resize = xmin * resize_w / orig_w
            xmax_resize = xmax * resize_w / orig_w
            ymin_resize = ymin * resize_h / orig_h
            ymax_resize = ymax * resize_h / orig_h

            # 2. Resize → CenterCrop
            xmin_crop = xmin_resize - crop_left
            xmax_crop = xmax_resize - crop_left
            ymin_crop = ymin_resize - crop_top
            ymax_crop = ymax_resize - crop_top

            # 3. check bbox collision crop
            if xmax_crop > 0 and xmin_crop < crop_size and ymax_crop > 0 and ymin_crop < crop_size:
                # crop clipping
                xmin_final = max(0, xmin_crop)
                xmax_final = min(crop_size, xmax_crop)
                ymin_final = max(0, ymin_crop)
                ymax_final = min(crop_size, ymax_crop)

                rect = Rectangle((xmin_final, ymin_final),
                               xmax_final - xmin_final,
                               ymax_final - ymin_final,
                               linewidth=2, edgecolor='yellow', facecolor='none')
                axes[0, col].add_patch(rect)

    if col == 0:
        axes[0, col].set_ylabel("Original\nImage", fontsize=12, rotation=0, ha='right', va='center')

    for row, step in enumerate(vis_steps, start=1):
        step_idx = steps.index(step)
        model = models_steps[step_idx].to(device).eval()
        linear = linear_steps[step_idx].to(device).eval()

        target_layers = [model.layer4[-1]]
        model_grad = nn.Sequential(model, linear)

        img_tensor = orig_batch.unsqueeze(0).to(device)

        with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
            grayscale_cam = cam(input_tensor=img_tensor, targets=None)
            grayscale_cam = grayscale_cam[0, :]
            visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        cache = inference_cache[step][idx]
        obj_correct = cache['obj_top5']
        bg_correct = cache['bg_top5']
        orig_correct = cache['orig_top5']

        axes[row, col].imshow(visualization)
        axes[row, col].axis('off')

        border_color = 'green' if (orig_correct and obj_correct) else 'red'
        rect = Rectangle((0, 0), 1, 1, transform=axes[row, col].transAxes,
                         linewidth=4, edgecolor=border_color, facecolor='none')
        axes[row, col].add_patch(rect)

        o_symbol = 'O' if orig_correct else 'X'
        obj_symbol = 'O' if obj_correct else 'X'
        bg_symbol = 'O' if bg_correct else 'X'

        axes[row, col].set_title(f"Original:{o_symbol}, Obj Only:{obj_symbol}, BG Only:{bg_symbol}",
                                 fontsize=10, color=border_color)

        if col == 0:
            axes[row, col].set_ylabel(f"Step {step}", fontsize=11, rotation=0, ha='right', va='center')

plt.tight_layout()
# plt.savefig(f'gradcam_transition.pdf')
plt.show()

In [ ]:
# for test
sample_idx = 43568
image_path, _ = dataset_test.samples[sample_idx]

# original size
img = Image.open(image_path)
orig_w, orig_h = img.size

img_name = os.path.basename(image_path)
xml_name = os.path.splitext(img_name)[0] + ".xml"
xml_path = os.path.join(bbox_dir, xml_name)

# function ratio
calculated_ratio = get_bbox_area_ratio_after_crop(xml_path, orig_w, orig_h)

# real mask
orig_batch, object_batch, background_batch, classes_label, bbox_mask, _ = masking_dataset[sample_idx]
actual_bbox_mask = (bbox_mask[0] == 0)  #
actual_ratio = actual_bbox_mask.sum().item() / (224 * 224)

print(f"Calculated ratio: {calculated_ratio:.6f}")
print(f"Actual ratio from mask: {actual_ratio:.6f}")
print(f"Difference: {abs(calculated_ratio - actual_ratio):.6f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

img_display = denormalize(orig_batch)
img_np = img_display.permute(1, 2, 0).cpu().numpy()
axes[0].imshow(img_np)
axes[0].set_title("Original Image")

axes[1].imshow(actual_bbox_mask.cpu().numpy(), cmap='gray')
axes[1].set_title(f"Actual Mask (ratio={actual_ratio:.4f})")

masked_img = img_np.copy()
masked_img[~actual_bbox_mask.cpu().numpy()] = 0
axes[2].imshow(masked_img)
axes[2].set_title(f"BBox Only")

plt.tight_layout()
plt.show()